# 25 QC EEG Parcel Exports, Table S3, And Figures S2-S4

## What this notebook does

This notebook is the cleaned public-facing QC and manuscript-summary notebook for the Stage-2 EEG parcel exports.

It reads the current v3 QC CSV outputs, writes a manuscript-facing Table-S3 support CSV, and generates the cleaned figure files for Supplementary Figures S2-S4.

## Manuscript linkage

- Main Methods 2.2.3 support
- Supplementary Methods 1.3
- Supplementary Results 2.3
- Supplementary Table S3
- Supplementary Figures S2-S4

## Important scope note

This notebook uses only the current v3 exporter/QC schema. It does **not** reuse the older exploratory MAT-parsing cells from `r01_figs_eeg_parcel_pc.ipynb` that assumed older field names such as `*_parcelPC.mat`, `n_dipoles`, or `parcel_names`.

## Required upstream files

Before running this notebook, Stage 2 parcel export must already be complete and the current MATLAB QC helpers must already have written their CSV outputs.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# User-editable settings
# ------------------------------------------------------------------
parcel_output_dir = Path(r"C:\SET_STAGE2_PARCEL_OUTPUT_DIR")

figures_dir = parcel_output_dir / "figures_s2_s4"
table_s3_csv = parcel_output_dir / "table_s3_eeg_parcel_extraction_summary.csv"

# Extra QC sidecars that are useful to keep, even though they are not
# the numbered supplementary figures.
write_extra_qc_figures = True
top_n_low_pve_parcels = 20


In [ ]:
def require_file(path: Path) -> Path:
    if not path.exists():
        helper_message = "\n".join([
            "Missing required QC CSV:",
            f"  {path}",
            "",
            "Run the Stage-2 MATLAB exporter QC helpers first, for example:",
            "  r01_qc_v3_run_timeseries_and_gain_summary(outDir, 'PreferGNORM', true, 'ChunkRows', 20000);",
            "  r01_qc_v3_sign_convention_parcelpc(outDir, 'PreferGNORM', true, 'ParcelsPerRun', 25, 'CorrThr', 0.99);",
            "  [RUNSUM, PARCSUM] = r01_qc_v3_pve1_hist_and_lowparcels(outDir, 'PreferGNORM', true, 'BottomFrac', 0.05);",
        ])
        raise FileNotFoundError(helper_message)
    return path


def run_label_from_tag(run_tag: str) -> str:
    run_tag = str(run_tag)
    if "_desc-" in run_tag:
        return run_tag.split("_desc-", 1)[0]
    return run_tag


figures_dir.mkdir(parents=True, exist_ok=True)

gain_csv = require_file(parcel_output_dir / "qc_v3" / "qc_run_timeseries_gain_summary.csv")
sign_csv = require_file(parcel_output_dir / "qc_v3_sign" / "qc_sign_v3_summary.csv")
pve_run_csv = require_file(parcel_output_dir / "batch_pve1_run_quantiles_v3.csv")
pve_hist_csv = require_file(parcel_output_dir / "batch_pve1_histogram_v3.csv")
pve_lowfreq_csv = require_file(parcel_output_dir / "batch_pve1_lowparcels_frequency_named_v3.csv")

gain_df = pd.read_csv(gain_csv)
sign_df = pd.read_csv(sign_csv)
pve_run_df = pd.read_csv(pve_run_csv)
pve_hist_df = pd.read_csv(pve_hist_csv)
pve_lowfreq_df = pd.read_csv(pve_lowfreq_csv)

display(gain_df.head())


In [ ]:
# ------------------------------------------------------------------
# Build the manuscript-facing Table S3 support CSV
# ------------------------------------------------------------------
table_s3 = (
    gain_df[["runTag", "valid_parcels", "nan_frac_pc1", "pc1_std_median"]]
    .merge(sign_df[["runTag", "pass_rate", "corr_tcfix_median"]], on="runTag", how="left")
    .merge(
        pve_run_df[["runTag", "pve1_q10", "pve1_q50", "pve1_q90", "frac_pve1_lt20"]],
        on="runTag",
        how="left",
    )
)

table_s3["Run"] = table_s3["runTag"].map(run_label_from_tag)

table_s3 = table_s3.rename(
    columns={
        "valid_parcels": "Valid parcels",
        "nan_frac_pc1": "NaN fraction (PC1)",
        "pc1_std_median": "Median PC1 scale after gnorm",
        "pass_rate": "Sign-check pass rate",
        "corr_tcfix_median": "Median sign-check correlation",
        "pve1_q10": "PVE1 q10",
        "pve1_q50": "PVE1 q50",
        "pve1_q90": "PVE1 q90",
        "frac_pve1_lt20": "Fraction of parcels with PVE1 < 0.20",
    }
)

table_s3 = table_s3[[
    "Run",
    "Valid parcels",
    "NaN fraction (PC1)",
    "Median PC1 scale after gnorm",
    "Sign-check pass rate",
    "Median sign-check correlation",
    "PVE1 q10",
    "PVE1 q50",
    "PVE1 q90",
    "Fraction of parcels with PVE1 < 0.20",
]]

table_s3 = table_s3.sort_values("Run").reset_index(drop=True)
table_s3.to_csv(table_s3_csv, index=False)

print("Wrote Table-S3 support CSV:", table_s3_csv)
display(table_s3)


In [ ]:
# ------------------------------------------------------------------
# Figure S2. Run-wise median EEG parcel-PC1 scale after gain normalization
# ------------------------------------------------------------------
plot_df = gain_df.copy()
plot_df["Run"] = plot_df["runTag"].map(run_label_from_tag)
plot_df = plot_df.sort_values("Run")

fig_s2 = figures_dir / "fig_s2_gain_pc1_scale_after_gnorm.png"

plt.figure(figsize=(10, 4))
plt.plot(plot_df["Run"], plot_df["pc1_std_median"], marker="o")
plt.xticks(rotation=60, ha="right")
plt.ylabel("Median PC1 scale after gnorm")
plt.title("Figure S2. Run-wise median EEG parcel-PC1 scale after gain normalization")
plt.tight_layout()
plt.savefig(fig_s2, dpi=200)
plt.close()

print("Wrote:", fig_s2)


In [ ]:
# ------------------------------------------------------------------
# Figure S3. Pooled distribution of PVE1 across runs and parcels
# ------------------------------------------------------------------
fig_s3 = figures_dir / "fig_s3_pve1_histogram_pooled.png"

plt.figure(figsize=(7, 4))
plt.bar(pve_hist_df["pve1_bin_center"], pve_hist_df["count"], width=0.02)
plt.xlabel("PVE1 (PC1 / trace(Cp))")
plt.ylabel("Count")
plt.title("Figure S3. Pooled distribution of PVE1 across runs and parcels")
plt.tight_layout()
plt.savefig(fig_s3, dpi=200)
plt.close()

print("Wrote:", fig_s3)


In [ ]:
# ------------------------------------------------------------------
# Figure S4. Run-wise PVE1 quantiles
# ------------------------------------------------------------------
plot_df = pve_run_df.copy()
plot_df["Run"] = plot_df["runTag"].map(run_label_from_tag)
plot_df = plot_df.sort_values("Run")

fig_s4 = figures_dir / "fig_s4_pve1_run_quantiles.png"

plt.figure(figsize=(10, 4))
plt.plot(plot_df["Run"], plot_df["pve1_q50"], marker="o", label="q50")
plt.plot(plot_df["Run"], plot_df["pve1_q10"], marker="o", label="q10")
plt.plot(plot_df["Run"], plot_df["pve1_q90"], marker="o", label="q90")
plt.xticks(rotation=60, ha="right")
plt.ylabel("PVE1")
plt.title("Figure S4. Run-wise PVE1 quantiles")
plt.legend()
plt.tight_layout()
plt.savefig(fig_s4, dpi=200)
plt.close()

print("Wrote:", fig_s4)


In [ ]:
# ------------------------------------------------------------------
# Extra QC sidecars preserved from the earlier mixed notebook logic
# ------------------------------------------------------------------
if write_extra_qc_figures:
    sign_plot_df = sign_df.copy()
    sign_plot_df["Run"] = sign_plot_df["runTag"].map(run_label_from_tag)
    sign_plot_df = sign_plot_df.sort_values("Run")

    sign_fig = figures_dir / "qc_sign_pass_rate_per_run.png"
    plt.figure(figsize=(10, 4))
    plt.plot(sign_plot_df["Run"], sign_plot_df["pass_rate"], marker="o")
    plt.xticks(rotation=60, ha="right")
    plt.ylim(-0.05, 1.05)
    plt.ylabel("Pass rate")
    plt.title("QC sidecar. Sign-convention pass rate per run")
    plt.tight_layout()
    plt.savefig(sign_fig, dpi=200)
    plt.close()
    print("Wrote:", sign_fig)

    lowfreq_plot_df = pve_lowfreq_df.sort_values("freq_bottomFrac", ascending=False).head(top_n_low_pve_parcels)
    lowfreq_fig = figures_dir / "qc_low_pve_parcel_frequency_top20.png"

    plt.figure(figsize=(8, 6))
    plt.barh(lowfreq_plot_df["parcel_name"], lowfreq_plot_df["freq_bottomFrac"])
    plt.gca().invert_yaxis()
    plt.xlabel("Frequency of appearing in the bottom-PVE set")
    plt.title("QC sidecar. Parcels most often in the bottom-PVE set")
    plt.tight_layout()
    plt.savefig(lowfreq_fig, dpi=200)
    plt.close()
    print("Wrote:", lowfreq_fig)


## Notes

- The MATLAB QC helpers remain the source of truth for run-level parcel-export diagnostics.
- This notebook intentionally consumes their CSV outputs instead of re-implementing the parcel-level math in Python.
- The Stage-2 parcel exporter still writes additional provenance outputs such as PC2 and extra sidecars; those are preserved even though the manuscript-facing feature is PC1.
